<a href="https://colab.research.google.com/github/piaoruilin/hallucination-medical-vlm/blob/main/stage1_trivial_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1: trivial baselines for hallucination detection
Scores simple detectors on your Qwen-injected data (word-level labels), so you know the floor a real detector must beat.

**Part A (cells 2-6) needs no GPU.** Part B (cells 7-9) loads Qwen as an LLM judge and needs a GPU.
All scores are for the *hallucinated* class: precision, recall, F1 (not plain accuracy, since most words are label 0).

In [1]:
!pip install -q "bitsandbytes>=0.46.1" accelerate transformers tqdm
# If this installs something new, do Runtime > Restart session once, then skip this cell.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.6 MB/s eta 0:00:00


In [9]:
from google.colab import drive
drive.mount("/content/drive")

import os, glob, json
OUT_DIR = "/content/drive/MyDrive/llava_pmcvqa_lora"
print("folder exists:", os.path.exists(OUT_DIR))
print(os.listdir(OUT_DIR) if os.path.exists(OUT_DIR) else "")
print("final adapter:", os.path.exists(OUT_DIR + "/final/adapter_model.safetensors"))

ckpts = sorted(glob.glob(OUT_DIR + "/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
if ckpts:
    st = json.load(open(ckpts[-1] + "/trainer_state.json"))
    print("last checkpoint: step", st["global_step"], "of", st["max_steps"])
    losses = [(h["step"], round(h["loss"], 3)) for h in st["log_history"] if "loss" in h]
    print(losses[:3], "...", losses[-3:])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
folder exists: True
[]
final adapter: False


## Part A: load data and score the no-model baselines

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import re, json, random, numpy as np, pandas as pd

JSONL_PATH = "/content/drive/MyDrive/halluc_injected.jsonl"
SAVE_DIR   = "/content/drive/MyDrive/halluc_baselines"
BAD_ROWS   = []     # put the `row` ids of any pairs you judged bad after reading your sample

d = pd.read_json(JSONL_PATH, lines=True)
d = d[~d["row"].isin(BAD_ROWS)].reset_index(drop=True)

assert all(len(t) == len(l) for t, l in zip(d["tokens"], d["labels"])), "tokens and labels differ in length"
n_words = int(d["tokens"].apply(len).sum())
n_pos = int(d["labels"].apply(sum).sum())
print(len(d), "samples |", n_words, "words |", n_pos, "hallucinated words (%.1f%%)" % (100 * n_pos / n_words))
print(d["hallucinated"].value_counts().to_dict())
print(d["type"].value_counts().to_dict())

154 samples | 2086 words | 196 hallucinated words (9.4%)
{1: 77, 0: 77}
{'invented_finding': 56, 'wrong_side': 36, 'wrong_attribute': 32, 'wrong_organ': 30}


In [4]:
def norm_word(w):
    return re.sub(r"[^a-z0-9]", "", str(w).lower())

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

def evaluate(df, col):
    # word level: all words from all samples pooled
    tp = fp = fn = 0
    for labels, pred in zip(df["labels"], df[col]):
        for l, p in zip(labels, pred):
            tp += int(l == 1 and p == 1)
            fp += int(l == 0 and p == 1)
            fn += int(l == 1 and p == 0)
    wP, wR, wF = prf(tp, fp, fn)
    # sample level: a sample counts as "flagged" if any word is flagged
    y = df["hallucinated"].values
    yhat = df[col].apply(lambda p: int(any(p))).values
    stp = int(((y == 1) & (yhat == 1)).sum())
    sfp = int(((y == 0) & (yhat == 1)).sum())
    sfn = int(((y == 1) & (yhat == 0)).sum())
    sP, sR, sF = prf(stp, sfp, sfn)
    return {"word_P": wP, "word_R": wR, "word_F1": wF,
            "sample_P": sP, "sample_R": sR, "sample_F1": sF, "sample_acc": float((y == yhat).mean())}

def by_type(df, col):
    rows = []
    for t, g in df[df["hallucinated"] == 1].groupby("type"):
        tp = sum(sum(1 for l, p in zip(L, P) if l == 1 and p == 1) for L, P in zip(g["labels"], g[col]))
        pos = sum(sum(L) for L in g["labels"])
        rows.append({"type": t, "n": len(g),
                     "word_recall": tp / pos if pos else 0.0,
                     "sample_detect_rate": float(g[col].apply(lambda p: int(any(p))).mean())})
    return pd.DataFrame(rows)

results = {}

In [5]:
# 1. flag nothing
d["pred_none"] = d["tokens"].apply(lambda t: [0] * len(t))
results["flag nothing"] = evaluate(d, "pred_none")

# 2. flag every word
d["pred_all"] = d["tokens"].apply(lambda t: [1] * len(t))
results["flag everything"] = evaluate(d, "pred_all")

# 3. flag random words at the true hallucination rate (averaged over 20 seeds)
rate = n_pos / n_words
runs = []
for seed in range(20):
    rng = np.random.default_rng(seed)
    d["_p"] = d["tokens"].apply(lambda t: (rng.random(len(t)) < rate).astype(int).tolist())
    runs.append(evaluate(d, "_p"))
d = d.drop(columns="_p")
results["random (matched rate)"] = pd.DataFrame(runs).mean().to_dict()

pd.DataFrame(results).T.round(3)

,word_P,word_R,word_F1,sample_P,sample_R,sample_F1,sample_acc
flag nothing,0.000,0.000,0.000,0.000,0.000,0.000,0.500
flag everything,0.094,1.000,0.172,0.500,1.000,0.667,0.500
random (matched rate),0.097,0.098,0.098,0.518,0.736,0.608,0.525


**Shortcut check.** A detector trained on injected data can cheat by learning *which kinds of words* Qwen likes to swap (sides, organs, locations).
This baseline flags any word from a small anatomy/laterality list, with no model at all. If it scores well, your data has a shortcut.

In [6]:
LEX = {"left", "right", "upper", "lower", "middle", "anterior", "posterior", "superior", "inferior",
       "bilateral", "lateral", "medial", "lung", "lungs", "liver", "heart", "kidney", "spleen", "brain",
       "lobe", "lobes", "chest", "pleural", "mediastinal", "mediastinum", "small", "large", "mild", "severe"}

d["pred_lexicon"] = d["tokens"].apply(lambda t: [int(norm_word(w) in LEX) for w in t])
results["lexicon shortcut"] = evaluate(d, "pred_lexicon")
pd.DataFrame(results).T.round(3)

,word_P,word_R,word_F1,sample_P,sample_R,sample_F1,sample_acc
flag nothing,0.000,0.000,0.000,0.000,0.000,0.000,0.500
flag everything,0.094,1.000,0.172,0.500,1.000,0.667,0.500
random (matched rate),0.097,0.098,0.098,0.518,0.736,0.608,0.525
lexicon shortcut,0.234,0.398,0.295,0.515,0.896,0.654,0.526


## Part B: LLM judge (needs a GPU)
Qwen reads the question and answer and lists the words it thinks are wrong. It **cannot see the image**, so this is a text-only baseline.
`MODES` can also include `"reference"`, where Qwen is shown the correct answer too. That is an easy "oracle" setting, useful as a check on your labels, not as a real baseline.

In [7]:
import torch
print("GPU available:", torch.cuda.is_available())
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"   # use the same Qwen as your injection step
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(model_id)
llm = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
llm.eval()

GPU available: True


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [8]:
SYSTEM = ("You check answers to medical image questions for hallucinated (incorrect) content. "
          "You cannot see the image. You output only valid JSON.")

def build_prompt(q, a, ref):
    s = "Question: " + str(q) + "\n"
    if ref is not None:
        s += "Reference (correct) answer: " + str(ref) + "\n"
    s += "Answer to check: " + str(a) + "\n\n"
    if ref is None:
        s += ("List the words in the answer that are likely hallucinated: an incorrect organ, side, "
              "location, finding, size or attribute. Copy the words exactly as written.\n")
    else:
        s += ("List the words in the answer that contradict or are not supported by the reference answer. "
              "Copy the words exactly as written.\n")
    s += 'Return only JSON like {"hallucinated_words": ["word1", "word2"]}, or {"hallucinated_words": []} if nothing is wrong.'
    return s

def ask_judge(q, a, ref):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": build_prompt(q, a, ref)}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=80, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

def flags_from_llm(tokens, words):
    if isinstance(words, str):
        words = [words]
    S = {norm_word(x) for w in words for x in str(w).split()} - {""}
    return [int(norm_word(t) in S) for t in tokens]

MODES = ["no_reference"]      # add "reference" for the oracle check (doubles the run time)

for mode in MODES:
    preds, raws, bad = [], [], 0
    for _, r in tqdm(d.iterrows(), total=len(d)):
        raw = ask_judge(r["question"], r["answer"], r["gt_answer"] if mode == "reference" else None)
        o = parse_json(raw)
        if not isinstance(o, dict):
            bad += 1
            words = []
        else:
            words = o.get("hallucinated_words", [])
        preds.append(flags_from_llm(r["tokens"], words))
        raws.append(raw)
    d["pred_" + mode] = preds
    d["raw_" + mode] = raws
    results["Qwen judge (" + mode + ")"] = evaluate(d, "pred_" + mode)
    print(mode, "unparseable outputs:", bad, "of", len(d))

  0%|          | 0/154 [00:00<?, ?it/s]

no_reference unparseable outputs: 0 of 154


## Results

In [10]:
table = pd.DataFrame(results).T.round(3)
display(table)

for mode in MODES:
    print("\nBy hallucination type, Qwen judge (" + mode + ")")
    display(by_type(d, "pred_" + mode).round(3))

,word_P,word_R,word_F1,sample_P,sample_R,sample_F1,sample_acc
flag nothing,0.000,0.000,0.000,0.000,0.000,0.000,0.500
flag everything,0.094,1.000,0.172,0.500,1.000,0.667,0.500
random (matched rate),0.097,0.098,0.098,0.518,0.736,0.608,0.525
lexicon shortcut,0.234,0.398,0.295,0.515,0.896,0.654,0.526
Qwen judge (no_reference),0.344,0.265,0.300,0.649,0.649,0.649,0.649



By hallucination type, Qwen judge (no_reference)


,type,n,word_recall,sample_detect_rate
0,invented_finding,28,0.186,0.750
1,wrong_attribute,16,0.065,0.312
2,wrong_organ,15,0.483,0.733
3,wrong_side,18,0.652,0.722


In [11]:
# Look at examples: gold hallucinated words vs what the judge flagged
col = "pred_" + MODES[0]
for _, r in d[d["hallucinated"] == 1].sample(10, random_state=0).iterrows():
    gold = [t for t, l in zip(r["tokens"], r["labels"]) if l]
    flagged = [t for t, p in zip(r["tokens"], r[col]) if p]
    print(r["answer"])
    print("   gold:", gold, "| flagged:", flagged)

The chest X-ray showed patchy opacifications with fibrotic shadows in the lower left lung.
   gold: ['lower', 'left'] | flagged: ['lower', 'left']
The previous chest radiography was performed 4 years ago and showed an enlarged cardiothoracic ratio and left pulmonary artery, and a new kidney tumor.
   gold: ['new', 'kidney', 'tumor.'] | flagged: ['kidney']
The images being compared in this caption are from Day 15, which shows significant improvement over Day 22.
   gold: ['Day', '15,', 'which', 'shows', 'significant', 'improvement'] | flagged: ['Day', '15,', 'Day', '22.']
The image shows an enlarged intraoperative visual field in the upper mediastinum.
   gold: ['enlarged'] | flagged: ['intraoperative']
The pleura was thinner in the lower part.
   gold: ['lower'] | flagged: ['thinner']
The chest CT scan shows a primary lesion in the right upper lobe and a small pleural effusion.
   gold: ['pleural'] | flagged: []
The patient's right lung is affected, and there is also a small tumor on t

In [12]:
import os
os.makedirs(SAVE_DIR, exist_ok=True)
table.to_csv(SAVE_DIR + "/stage1_results.csv")
d.to_json(SAVE_DIR + "/stage1_predictions.jsonl", orient="records", lines=True)
print("saved to", SAVE_DIR)

saved to /content/drive/MyDrive/halluc_baselines
